# Recipe Preference Matcher — MLP

Trains a tiny on-device MLP that scores how well a TheMealDB recipe matches a user’s saved cuisine area and category preferences.

**Input (4 floats):**
- `area_match` — 1.0 if recipe’s strArea is in user’s area prefs, else 0.0
- `cat_match` — 1.0 if recipe’s strCategory is in user’s category prefs, else 0.0
- `area_norm` — same as area_match (extensible in future to fraction of prefs matched)
- `cat_norm` — same as cat_match

**Output:** continuous score in [0, 1]

**Label rule (regression):**
- `y = area_match * 0.5 + cat_match * 0.5`
- both match → 1.0 (100% match badge, green)
- area or category only → 0.5 (50% match badge, secondary color)
- no match → 0.0 (no badge)


## Cell 1 — Config

In [ ]:
EPOCHS = 40
BATCH_SIZE = 64
VAL_SPLIT = 0.2
N_SAMPLES = 10_000

## Cell 2 — Vocabulary Manifest

These must match the values used in the React Native app (`account.tsx` preference options).
The model itself does not depend on this list at inference time — it only needs the binary match signals.

In [ ]:
# TheMealDB canonical strArea values
AREAS = [
    "American", "British", "Canadian", "Chinese", "Croatian",
    "Dutch", "Egyptian", "Filipino", "French", "Greek",
    "Indian", "Irish", "Italian", "Jamaican", "Japanese",
    "Kenyan", "Malaysian", "Mexican", "Moroccan", "Polish",
    "Portuguese", "Russian", "Spanish", "Thai", "Tunisian",
    "Turkish", "Ukrainian", "Unknown", "Vietnamese",
]

# TheMealDB canonical strCategory values
CATEGORIES = [
    "Beef", "Breakfast", "Chicken", "Dessert", "Goat",
    "Lamb", "Miscellaneous", "Pasta", "Pork", "Seafood",
    "Side", "Starter", "Vegan", "Vegetarian",
]

N_AREAS = len(AREAS)          # 29
N_CATEGORIES = len(CATEGORIES) # 14
print(f"Areas: {N_AREAS}, Categories: {N_CATEGORIES}")

## Cell 3 — Training Data Generation

Synthetic data: simulate random (recipe, user_prefs) pairs and label them based on match quality.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def generate_sample():
    # Simulate a recipe: one area, one category
    recipe_area_idx = rng.integers(0, N_AREAS)
    recipe_cat_idx  = rng.integers(0, N_CATEGORIES)

    # Simulate user preferences: 0–5 areas, 0–5 categories
    n_user_areas = rng.integers(0, 6)
    n_user_cats  = rng.integers(0, 6)
    user_areas   = set(rng.choice(N_AREAS, n_user_areas, replace=False).tolist())
    user_cats    = set(rng.choice(N_CATEGORIES, n_user_cats, replace=False).tolist())

    # Binary match signals
    area_match = 1.0 if recipe_area_idx in user_areas else 0.0
    cat_match  = 1.0 if recipe_cat_idx  in user_cats  else 0.0

    # 4-float input vector
    # Features 0,1: direct match signals
    # Features 2,3: same for now (extensible — could become fraction of prefs matched)
    x = np.array([area_match, cat_match, area_match, cat_match], dtype=np.float32)

    # Label: continuous regression target
    # both match → 1.0, area-only or cat-only → 0.5, no match → 0.0
    y = area_match * 0.5 + cat_match * 0.5

    return x, y


samples = [generate_sample() for _ in range(N_SAMPLES)]
X_train = np.stack([s[0] for s in samples])
y_train = np.array([s[1] for s in samples], dtype=np.float32)

print(f"X_train shape: {X_train.shape}")
print(f"Label distribution: 0.0={float((y_train==0).mean()):.2f}, 0.5={float((y_train==0.5).mean()):.2f}, 1.0={float((y_train==1.0).mean()):.2f}")


## Cell 4 — Class Balance Check

In [ ]:
from collections import Counter

# For regression labels, show value distribution
unique, cnts = np.unique(y_train, return_counts=True)
print("Label value counts:")
for v, c in zip(unique, cnts):
    print(f"  {v:.1f}: {c} ({100*c/len(y_train):.1f}%)")


## Cell 5 — Model Architecture

Tiny MLP: 4 inputs → 8 hidden (ReLU) → 1 output (Sigmoid).  
Total parameters: (4×8 + 8) + (8×1 + 1) = **49 weights** → < 5 KB TFJS shard.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

model = tf.keras.Sequential([
    layers.Dense(8, activation="relu", input_shape=(4,)),
    layers.Dense(1, activation="linear"),
], name="preference_mlp")

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"],
)

model.summary()


## Cell 6 — Training

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=[early_stopping],
)

print(f"
Final val_loss: {min(history.history['val_loss']):.4f}")
print(f"Final val_mae:  {min(history.history['val_mae']):.4f}")


## Cell 7 — Smoke Tests

Verify the model learned the correct direction. Expected (regression targets):
- Both match → ~1.0
- Area-only match → ~0.5
- Category-only match → ~0.5
- No match → ~0.0


In [ ]:
test_cases = [
    ([1.0, 1.0, 1.0, 1.0], "both match    → expect ~1.0"),
    ([1.0, 0.0, 1.0, 0.0], "area only     → expect ~0.5"),
    ([0.0, 1.0, 0.0, 1.0], "category only → expect ~0.5"),
    ([0.0, 0.0, 0.0, 0.0], "no match      → expect ~0.0"),
]

for x_vals, label in test_cases:
    x = np.array([x_vals], dtype=np.float32)
    score = float(np.clip(model.predict(x, verbose=0)[0][0], 0.0, 1.0))
    print(f"{label}  →  score = {score:.3f}")


## Cell 8 — Save Keras Model

In [ ]:
model.save("preference_mlp.keras")
print("Saved: preference_mlp.keras")

## Cell 9 — Convert to TFJS

Run this cell to produce `./tfjs_preference_mlp/model.json` and a single small `.bin` weight shard.  
Then copy the directory contents into `SpisNemt_FE/src/ml/MLP/`.

**Prerequisite:** `pip install tensorflowjs`

In [ ]:
import tensorflowjs as tfjs
import os

# Create the output directory if it doesn't exist
output_path = './tmp/mlp'
if not os.path.exists(output_path):
    os.makedirs(output_path)

# Convert the model directly from the memory object
# This avoids the "InputLayer" deserialization bug in the CLI
tfjs.converters.save_keras_model(model, output_path)

print(f"Conversion complete. Files in {output_path}:")
!ls -lh {output_path}

## Cell 10 — Next Steps

After the converter runs:

1. Copy `./tfjs_preference_mlp/model.json` and `./tfjs_preference_mlp/group1-shard1of1.bin` into `SpisNemt_FE/src/ml/MLP/`
2. The TypeScript service at `SpisNemt_FE/src/services/ml/preferencesMatcher.ts` will load them via `bundleResourceIO`
3. No code changes needed — asset paths are already wired up